In [1]:
import scMPRAforge as scm
from dask_jobqueue import SLURMCluster
from dask.distributed import Client, LocalCluster
import pandas as pd

2025-12-08 17:15:22.133333: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-08 17:15:22.136392: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
from pathlib import Path

In [3]:
DATA_ROOT=Path("/nfs/roberts/project/pi_skr2/shared/tabula_data")
path=DATA_ROOT/"simulated/shendure_pow_analysis"
name="sim_with_orthos_20251206"

In [4]:
spread_hypothesis_20251119=scm.HypothesisSet.from_tsv(path/"spread_hypothesis_20251119.tsv")

In [5]:
df = spread_hypothesis_20251119.df
by_ct = df[df.comparison_CRE!=df.reference_CRE]
by_cre = df[df.comparison_CRE==df.reference_CRE]

by_ct.to_csv(path/"CT_spread_hypothesis_20251119.tsv", sep='\t')
by_cre.to_csv(path/"CRE_spread_hypothesis_20251119.tsv", sep='\t')

In [6]:
ct_spread_hypothesis_20251119=scm.HypothesisSet.from_tsv(path/"CT_spread_hypothesis_20251119.tsv")

In [7]:
from dask.distributed import get_client#Semaphore, as_completed,

In [8]:
ortho_root=path/name/"orthos_with_precomputed_wald"
output_root=path/name/"results"
output_root.mkdir(exist_ok=True,parents=True)
input_ortho_names=[path.name for path in ortho_root.iterdir()]

#Semaphore(max_leases=5, name="test")

def compute_one_test(input_root, name, output_root, hypothesis_set, hypothesis_set_name, test_type, use_client=False):
    #sem = Semaphore(name="test")
    #with sem:
    client=get_client()
    ortho_oi=scm.ortho.load(client=client,
                                path=input_root,
                                name=name)
    scmpradat_oi=scm.scMPRA_data.from_parquet(path/"sim_with_orthos_20251206"/"scMPRA"/f"{name}.scmpra")
    ortho_oi.training_data=scmpradat_oi
    print(dir(ortho_oi))
    runner = scm.HypothesisTester(test_type)
    output_short=Path(output_root)/hypothesis_set_name/test_type
    output_short.mkdir(exist_ok=True,parents=True)
    if not use_client:
        runner.run(hypothesis_set, ortho_oi).to_tsv(output_short/name)
    else:
        runner.run(hypothesis_set, ortho_oi, client).to_tsv(output_short/name)
    #runner.run(hypothesis_set, scmpradat_oi, client).to_tsv(output_short/name)

In [9]:
local=True
if local:
    cluster=LocalCluster(memory_limit='48G')
    client = Client(cluster)
else:
    cluster=SLURMCluster(
        cores=4,#cores per slurm job
        memory="64G",#memory per slurm job
        processes=1,#dask workers per slurm job
        job_extra_directives=["-p day", 
            f"--job-name=simclust_worker",
            f"--time=2:00:00",
            f"--output=worker_%j.out"]
    )
    cluster.scale(jobs=1)
    client = Client(cluster,
            timeout=f"{10*60}s",   # Client <-> scheduler timeout 
            heartbeat_interval="20s",  # Worker heartbeat interval,
        )

In [10]:
client.dashboard_link

'http://127.0.0.1:8787/status'

In [ ]:
a = client.submit(compute_one_test,
                       input_root=ortho_root,
                       name='0',
                       output_root=output_root,
                       hypothesis_set=ct_spread_hypothesis_20251119,
                       hypothesis_set_name="spread_hypothesis_20251119",
                       test_type="wald", use_client=True)

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nb_versus_means', 'annotate_models', 'by_cell_type', 'by_cell_type_design', 'by_cell_type_parameters', 'by_cre', 'by_cre_design', 'by_cre_parameters', 'clean', 'compute_model_qc', 'criss_cross', 'extract_params', 'load', 'make_wald_eval_bundle', 'precompute_wald', 'save', 'training_data', 'wald_precomp']


In [13]:
a.result

<bound method Future.result of <Future: pending, key: compute_one_test-9fa6878af78828e950dc75701ba600df>>

In [ ]:
futures_wald = [client.submit(compute_one_test,
                       input_root=ortho_root,
                       name=name_oi,
                       output_root=output_root,
                       hypothesis_set=ct_spread_hypothesis_20251119,
                       hypothesis_set_name="spread_hypothesis_20251119",
                       test_type="wald") for name_oi in input_ortho_names]

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nb_versus_means', 'annotate_models', 'by_cell_type', 'by_cell_type_design', 'by_cell_type_parameters', 'by_cre', 'by_cre_design', 'by_cre_parameters', 'clean', 'compute_model_qc', 'criss_cross', 'extract_params', 'load', 'make_wald_eval_bundle', 'precompute_wald', 'save', 'training_data', 'wald_precomp']
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_

In [31]:
futures_wald

[<Future: finished, type: NoneType, key: compute_one_test-5e3e763195cdd6a771cacfe64fd674bb>,
 <Future: pending, key: compute_one_test-69d9901342520e0975affb32efe0a680>,
 <Future: pending, key: compute_one_test-02da806b789aac25550480312ec1bbd1>,
 <Future: pending, key: compute_one_test-2ec3cdc8ee6e655e89e553e815ba57d1>,
 <Future: pending, key: compute_one_test-e2d7e557a6bce814c7b2c192d502a760>]

In [ ]:
futures_mwu = [client.submit(compute_one_test,
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=spread_hypothesis_20251119,
                        hypothesis_set_name="spread_hypothesis_20251119",
                        test_type="mwu", use_client=True) for name_oi in input_ortho_names]

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nb_versus_means', 'annotate_models', 'by_cell_type', 'by_cell_type_design', 'by_cell_type_parameters', 'by_cre', 'by_cre_design', 'by_cre_parameters', 'clean', 'compute_model_qc', 'criss_cross', 'extract_params', 'load', 'make_wald_eval_bundle', 'precompute_wald', 'save', 'training_data', 'wald_precomp']
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_

In [10]:
results = [compute_one_test(
                        input_root=ortho_root,
                        name=name_oi,
                        output_root=output_root,
                        hypothesis_set=spread_hypothesis_20251119,
                        hypothesis_set_name="spread_hypothesis_20251119",
                        test_type="mwu") for name_oi in input_ortho_names]

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_nb_versus_means', 'annotate_models', 'by_cell_type', 'by_cell_type_design', 'by_cell_type_parameters', 'by_cre', 'by_cre_design', 'by_cre_parameters', 'clean', 'compute_model_qc', 'criss_cross', 'extract_params', 'load', 'make_wald_eval_bundle', 'precompute_wald', 'save', 'training_data', 'wald_precomp']
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_

In [ ]:
futures_mwu[0].result()

2025-12-08 15:51:40,954 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('compute_one_wald-2a680bb85a89ae8d9ca8ed4c015865f1')" coro=<Worker.execute() done, defined at /home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-12-08 15:51:40,955 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('compute_one_wald-8729066491ecd150a0961a0f1966eac1')" coro=<Worker.execute() done, defined at /home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-12-08 15:51:40,958 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('compute_one_wald-8bbf130eabfeb18117f599b2b8f83269')" coro=<Worker.execute() done, defined at /home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/distributed/worker_state_machin

KeyboardInterrupt: 

2025-12-08 15:51:41,828 - distributed.worker - WARNING - Scheduler was unaware of this worker; shutting down.
2025-12-08 15:51:41,829 - distributed.worker - WARNING - Scheduler was unaware of this worker; shutting down.
2025-12-08 15:51:41,830 - distributed.worker - WARNING - Scheduler was unaware of this worker; shutting down.
2025-12-08 15:51:41,926 - distributed.worker - WARNING - Scheduler was unaware of this worker; shutting down.


In [16]:
client.close()
cluster.close()

2025-12-08 17:14:27,062 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('compute_one_test-cfabaed90a31ded0f3060faafa8eac4b')" coro=<Worker.execute() done, defined at /home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-12-08 17:14:27,062 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('compute_one_test-dc6f04e2d4c1e5e26510aff7ee081df3')" coro=<Worker.execute() done, defined at /home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/distributed/worker_state_machine.py:3607>> ended with CancelledError
2025-12-08 17:14:27,063 - distributed.worker.state_machine - WARNING - Async instruction for <Task cancelled name="execute('compute_one_test-db8fc46e2b9ba7561fb2352328990c52')" coro=<Worker.execute() done, defined at /home/eng26/.conda/envs/scmpra/lib/python3.10/site-packages/distributed/worker_state_machin